# 11 — Revenue Impact Simulator

**Objective:** Simulate marketing campaign success rates (churn reduction percentages) and evaluate the ROI and net saved revenue of preventing customer churn.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, List

from src.config import *
from src.utils import logger, timer

## 1. Simulation and ROI Logic

Define simulation functions that estimate protected CLV by saving at-risk customers (prioritized by highest FVS), calculate expenses, and forecast ROI.

In [2]:
def simulate_retention_impact(
    df: pd.DataFrame,
    improvement_pcts: List[float] = [0.05, 0.10, 0.15, 0.20, 0.25],
) -> pd.DataFrame:
    churn_col = "Churn_Probability" if "Churn_Probability" in df.columns else "Churn"
    at_risk = df[df[churn_col] > 0.5].copy()
    total_at_risk = len(at_risk)
    total_at_risk_clv = at_risk["CLV"].sum()
    avg_clv = at_risk["CLV"].mean()

    if "FVS" in at_risk.columns:
        at_risk = at_risk.sort_values("FVS", ascending=False)

    logger.info(f"At-risk customers: {total_at_risk:,}")
    logger.info(f"Total at-risk CLV: ${total_at_risk_clv:,.0f}")

    scenarios = []
    for pct in improvement_pcts:
        customers_saved = int(total_at_risk * pct)
        
        if "FVS" in at_risk.columns:
            clv_retained = at_risk.head(customers_saved)["CLV"].sum()
        else:
            clv_retained = customers_saved * avg_clv

        revenue_protected = clv_retained

        avg_fvs_saved = (
            at_risk.head(customers_saved)["FVS"].mean()
            if "FVS" in at_risk.columns and not at_risk.head(customers_saved).empty
            else 0
        )

        scenarios.append({
            "Retention_Improvement": f"{pct*100:.0f}%",
            "Improvement_Pct": pct,
            "Customers_At_Risk": total_at_risk,
            "Customers_Saved": customers_saved,
            "CLV_Retained": round(clv_retained, 2),
            "Revenue_Protected": round(revenue_protected, 2),
            "Avg_CLV_Saved": round(clv_retained / max(customers_saved, 1), 2),
            "Avg_FVS_Saved": round(avg_fvs_saved, 1),
        })

    result = pd.DataFrame(scenarios)
    logger.info(f"\nSimulation results:\n{result.to_string(index=False)}")
    return result

def calculate_roi(
    simulation: pd.DataFrame,
    cost_per_customer: float = 150.0,
) -> pd.DataFrame:
    result = simulation.copy()
    result["Campaign_Cost"] = result["Customers_Saved"] * cost_per_customer
    result["Net_Revenue"] = result["Revenue_Protected"] - result["Campaign_Cost"]
    result["ROI"] = np.where(
        result["Campaign_Cost"] > 0,
        ((result["Net_Revenue"]/result["Campaign_Cost"]) * 100).round(1),
        0.0
    )
    result["ROI_Label"] = result["ROI"].apply(lambda x: f"{x:.0f}%")
    return result

## 2. Run Financial Simulation

Load customer data, execute retention scenario mapping, and output the ROI comparison matrix.

In [3]:
data_path = FEATURES_DIR / "customer_segmented.csv"
df = pd.read_csv(data_path)

simulation = simulate_retention_impact(df)
roi = calculate_roi(simulation)
print(roi)

2026-06-12 11:31:38 | airline_loyalty | INFO | At-risk customers: 2,883
2026-06-12 11:31:38 | airline_loyalty | INFO | Total at-risk CLV: $22,679,083
2026-06-12 11:31:38 | airline_loyalty | INFO | 
Simulation results:
Retention_Improvement  Improvement_Pct  Customers_At_Risk  Customers_Saved  CLV_Retained  Revenue_Protected  Avg_CLV_Saved  Avg_FVS_Saved
                   5%             0.05               2883              144    2593235.75         2593235.75       18008.58           58.3
                  10%             0.10               2883              288    4519453.24         4519453.24       15692.55           52.9
                  15%             0.15               2883              432    6296669.75         6296669.75       14575.62           49.3
                  20%             0.20               2883              576    7666605.74         7666605.74       13310.08           46.4
                  25%             0.25               2883              720    9042689.83    

  Retention_Improvement  Improvement_Pct  Customers_At_Risk  Customers_Saved  \
0                    5%             0.05               2883              144   
1                   10%             0.10               2883              288   
2                   15%             0.15               2883              432   
3                   20%             0.20               2883              576   
4                   25%             0.25               2883              720   

   CLV_Retained  Revenue_Protected  Avg_CLV_Saved  Avg_FVS_Saved  \
0    2593235.75         2593235.75       18008.58           58.3   
1    4519453.24         4519453.24       15692.55           52.9   
2    6296669.75         6296669.75       14575.62           49.3   
3    7666605.74         7666605.74       13310.08           46.4   
4    9042689.83         9042689.83       12559.29           44.1   

   Campaign_Cost  Net_Revenue      ROI ROI_Label  
0        21600.0   2571635.75  11905.7    11906%  
1       